In [ ]:
from pathlib import Path

In [ ]:
# TYPE = "finetuning"
TYPE = "linear_probing"

sorting_models = [
    # previous models
    # SSL
    # "your_path/save/fundus_ssl/logs/finetuning/retmae/mae_0.7",
    "your_path/save/fundus_ssl/logs/finetuning/retfound/retfound",
    # "your_path/save/fundus_ssl/logs/finetuning/retmae/hmi_0.01_0.4_m0.7",
    # "your_path/save/fundus_ssl/logs/finetuning/retmae/hmi_0.1_0.4_m0.7",
    # "your_path/save/fundus_ssl/logs/finetuning/retmae/hmi_1_0.4_m0.7",
    # Auxiliary Loss
    "your_path/save/fundus_ssl/logs/finetuning/urfound/urfound",
    "your_path/save/fundus_ssl/logs/finetuning/flair/flair",
    "your_path/save/fundus_ssl/logs/finetuning/keepfit/keepfit",
    "your_path/save/fundus_ssl/logs/finetuning/retclip/retclip",
    "your_path/save/fundus_ssl/logs/finetuning/vilref/vilref",
    "your_path/save/fundus_ssl/logs/finetuning/preti/preti",
    "your_path/save/fundus_ssl/logs/finetuning/urfound/urfound_no_stand",
    "your_path/save/fundus_ssl/logs/finetuning/flair/flair_no_stand",
    "your_path/save/fundus_ssl/logs/finetuning/keepfit/keepfit_no_stand",
    "your_path/save/fundus_ssl/logs/finetuning/retclip/retclip_no_stand",
    "your_path/save/fundus_ssl/logs/finetuning/vilref/vilref_no_stand",
    "your_path/save/fundus_ssl/logs/finetuning/preti/preti_no_stand",
    # ours
    # "your_path/save/fundus_ssl/logs/finetuning/retmae/gmi_0.01_m0.7",
    # "your_path/save/fundus_ssl/logs/finetuning/retmae/gmi_0.1_m0.7",
    # "your_path/save/fundus_ssl/logs/finetuning/retmae/gmi_1_m0.7",
    # "your_path/save/fundus_ssl/logs/finetuning/retmae/retmae_g1_h1_0.4_m0.7"
]

if TYPE == "linear_probing":
    sorting_models = [p.replace("finetuning", "linear_probing") for p in sorting_models]

In [ ]:
results_path = []
for model_path in sorting_models:
    p = Path(model_path)
    for f in p.glob("*metrics.csv"):
        results_path.append((model_path, f))
        print(f)
        break

In [ ]:
import os

import pandas as pd

# auroc와 auprc를 별도로 저장할 딕셔너리
auroc_data = []
auprc_data = []

for model_path, metrics_file in results_path:
    # method 이름: 경로의 마지막 폴더명
    method = model_path.split("/")[-1]

    # metrics.csv 파일 읽기
    df = pd.read_csv(metrics_file)

    # auroc 데이터 저장
    auroc_row = {"method": method}
    auprc_row = {"method": method}

    # 평균 계산을 위한 임시 리스트
    auroc_means = []
    auprc_means = []

    # 각 데이터셋에 대해 처리
    for idx, row in df.iterrows():
        dataset = row["dataset"]

        # auroc 값
        auroc_formatted = f"{row['auroc_mean']:.3f}±{row['auroc_std']:.3f}"
        auroc_row[dataset] = auroc_formatted
        auroc_means.append(row["auroc_mean"])

        # auprc 값
        auprc_formatted = f"{row['auprc_mean']:.3f}±{row['auprc_std']:.3f}"
        auprc_row[dataset] = auprc_formatted
        auprc_means.append(row["auprc_mean"])

    # 평균 계산 (std 없이 mean만)
    auroc_row["Average"] = f"{sum(auroc_means)/len(auroc_means):.3f}"
    auprc_row["Average"] = f"{sum(auprc_means)/len(auprc_means):.3f}"

    auroc_data.append(auroc_row)
    auprc_data.append(auprc_row)

# 데이터프레임으로 변환
auroc_df = pd.DataFrame(auroc_data)
auprc_df = pd.DataFrame(auprc_data)

# 컬럼 순서 재정렬
column_order = ["method", "IDRiD", "MESSIDOR2", "PAPILA", "REFUGE", "FIVES", "RFMiD2", "Average"]
auroc_df = auroc_df[column_order]
auprc_df = auprc_df[column_order]

# 저장 경로 설정
output_dir = Path(
    f"your_path/src/tasks/pretrain/experiment/downstream/{TYPE}"
)
output_dir.mkdir(parents=True, exist_ok=True)

# 파일 저장
auroc_path = output_dir / "auroc_results.csv"
auprc_path = output_dir / "auprc_results.csv"

auroc_df.to_csv(auroc_path, index=False)
auprc_df.to_csv(auprc_path, index=False)

print(f"AUROC 결과 저장: {auroc_path}")
print(f"\n{auroc_df}")
print(f"\nAUPRC 결과 저장: {auprc_path}")
print(f"\n{auprc_df}")